# 03 — Pydantic Schemas — Practical Examples

**Covers**: Pydantic v2 basics, nested models, validators, batch extraction, real-world schema patterns

In [ ]:
import os, json
from openai import OpenAI
from pydantic import BaseModel, Field, field_validator, model_validator
from typing import Optional, Literal
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o-mini'
print('✅ Setup complete')

## 1. Pydantic Basics — Define and Validate

In [ ]:
class PersonInfo(BaseModel):
    name: str
    age: int = Field(ge=0, le=150)
    email: str
    occupation: Optional[str] = None
    is_active: bool = True

# Manual creation shows how validation works
person = PersonInfo(name="Alice", age=30, email="alice@example.com", occupation="Engineer")
print(f"Name: {person.name}, Age: {person.age}, Active: {person.is_active}")
print(f"Schema (truncated):")
schema = PersonInfo.model_json_schema()
print(json.dumps(schema, indent=2)[:600])

## 2. LLM Extraction with Pydantic — Simple Schema

In [ ]:
class BookInfo(BaseModel):
    title: str
    author: str
    year: int = Field(ge=1000, le=2030)
    genre: Literal["fiction", "non-fiction", "science", "biography", "self-help", "technology", "other"]
    page_count: Optional[int] = None
    rating: Optional[float] = Field(default=None, ge=1.0, le=5.0)

books_raw = [
    "'Atomic Habits' by James Clear (2018) — self-help, 320 pages. People rate it 4.8/5.",
    "'Dune' by Frank Herbert, published 1965. A science fiction epic. Around 896 pages.",
    "'The Lean Startup' by Eric Ries (2011). Business/technology book, highly rated at 4.1 stars."
]

for text in books_raw:
    r = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[{"role": "user", "content": f"Extract book info: {text}"}],
        response_format=BookInfo
    )
    book = r.choices[0].message.parsed
    print(f"📚 {book.title} by {book.author} ({book.year}) | {book.genre} | ⭐{book.rating}")

## 3. Nested Pydantic Models — Complex Data

In [ ]:
class GeoLocation(BaseModel):
    city: str
    country: str
    region: Optional[str] = None

class SocialMedia(BaseModel):
    platform: str
    handle: str  # e.g., @username

class PublicFigure(BaseModel):
    full_name: str
    profession: str
    nationality: str
    birth_year: Optional[int] = None
    known_for: list[str] = Field(
        description="List of 2-4 things this person is known for",
        min_length=1, max_length=5
    )
    based_in: GeoLocation
    social_accounts: list[SocialMedia] = Field(default_factory=list)

bio = """
Elon Musk, born in 1971 in South Africa (now US citizen), is an entrepreneur and businessman
based in Austin, Texas, USA. He is known for founding SpaceX, acquiring Twitter (now X),
leading Tesla, and his work on Neuralink. His Twitter handle is @elonmusk.
"""

r = client.beta.chat.completions.parse(
    model=MODEL,
    messages=[{"role": "user", "content": f"Extract person info:\n{bio}"}],
    response_format=PublicFigure
)
figure = r.choices[0].message.parsed
rprint(figure.model_dump())
print(f"\nCity: {figure.based_in.city}, Country: {figure.based_in.country}")
print(f"Known for: {figure.known_for}")

## 4. Field Validators — Auto-Correct LLM Output

In [ ]:
import re
from pydantic import field_validator

class ContactCard(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    company: str

    @field_validator("email")
    @classmethod
    def normalize_email(cls, v: str) -> str:
        """Ensure email is lowercase and stripped."""
        return v.strip().lower()

    @field_validator("phone")
    @classmethod
    def clean_phone(cls, v: Optional[str]) -> Optional[str]:
        """Strip phone to digits + + and - only."""
        if v is None:
            return None
        cleaned = re.sub(r'[^\d+\-\s]', '', v).strip()
        return cleaned if cleaned else None

    @field_validator("name")
    @classmethod
    def title_case_name(cls, v: str) -> str:
        """Ensure name is title-cased."""
        return v.strip().title()

# Test with messy input from LLM
business_card_text = """
Meet JOHN DOE from TechCorp Inc.
Email: JOHN.DOE@TECHCORP.COM
Phone: (555) 123-4567
"""

r = client.beta.chat.completions.parse(
    model=MODEL,
    messages=[{"role": "user", "content": f"Extract contact:\n{business_card_text}"}],
    response_format=ContactCard
)
contact = r.choices[0].message.parsed
print(f"Name:    {contact.name!r}")
print(f"Email:   {contact.email!r}   (normalized to lowercase)")
print(f"Phone:   {contact.phone!r}  (cleaned)")
print(f"Company: {contact.company!r}")

## 5. Batch Extraction — List Wrapper Pattern

In [ ]:
# When you need to extract MULTIPLE items, wrap them in a 'container' model
class Expense(BaseModel):
    description: str
    amount: float = Field(ge=0)
    category: Literal["food", "transport", "accommodation", "entertainment", "other"]
    date: Optional[str] = None

class ExpenseReport(BaseModel):
    """Container for batch expense extraction."""
    expense_items: list[Expense]
    total_amount: float
    currency: str = "USD"
    report_period: Optional[str] = None

receipt_text = """
Business Trip - November 2024:
- Uber to airport: $28.50 (Nov 12)
- Lunch with client at The Grill: $145.00 (Nov 13)  
- Hotel Downtown Inn x2 nights: $320.00 (Nov 12-13)
- Conference dinner: $89.99 (Nov 13)
- Train back to office: $22.75 (Nov 14)
"""

r = client.beta.chat.completions.parse(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are an expense report processor."},
        {"role": "user",   "content": f"Extract all expenses:\n{receipt_text}"}
    ],
    response_format=ExpenseReport
)
report = r.choices[0].message.parsed
print(f"Period: {report.report_period} | Currency: {report.currency}")
print(f"{'Description':<35} {'Category':<15} {'Amount':>8}")
print("-" * 60)
for e in report.expense_items:
    print(f"{e.description:<35} {e.category:<15} ${e.amount:>7.2f}")
print("-" * 60)
print(f"{'TOTAL':<35} {'':15} ${report.total_amount:>7.2f}")

## 6. model_dump — Converting to Dict/JSON

In [ ]:
class TechStack(BaseModel):
    language: str
    frameworks: list[str]
    databases: list[str]
    cloud_provider: Optional[str] = None
    deployment: Literal["docker", "kubernetes", "serverless", "bare-metal", "other"] = "other"

jd_text = "We use Python with FastAPI and Django. PostgreSQL and Redis for data. Deployed on AWS EKS with Kubernetes."

r = client.beta.chat.completions.parse(
    model=MODEL,
    messages=[{"role": "user", "content": f"Extract tech stack: {jd_text}"}],
    response_format=TechStack
)
stack = r.choices[0].message.parsed

# Different serialization options
print("=== As dict ===")
print(stack.model_dump())
print("\n=== As JSON string ===")
print(stack.model_dump_json(indent=2))
print("\n=== Excluding None fields ===")
print(stack.model_dump(exclude_none=True))

## 📌 Summary

- **BaseModel** = define schema in Python → auto JSON Schema
- **Field()** = add constraints: `ge`, `le`, `min_length`, `max_length`, `description`
- **Literal** = enum-like fields — LLM must pick from fixed options
- **Nested models** = compose for complex hierarchical data
- **`field_validator`** = auto-clean/normalize LLM output
- **Wrapper model** = batch extraction of multiple items via `list[SubModel]`
- **`.model_dump()`** and **`.model_dump_json()`** = convert back to dict/JSON